# 004 · The operator Module

> **Standard Library › Functional Tools** · Python 3.14

In C# you can't pass `+` around as a value — you write `(a, b) => a + b`. Python's `operator` module gives you **every operator as a ready-made function**, plus small factories (`itemgetter`, `attrgetter`, `methodcaller`) that build fast key functions for sorting, grouping and mapping.

**In this notebook you will learn:**

- `itemgetter`, `attrgetter` and `methodcaller` as key functions
- the function versions of operators (`add`, `mul`, `neg`, `contains`, ...) for `reduce`, `map` and `accumulate`
- what `operator.index` does and why it exists
- when a plain `lambda` is the better choice

## .NET mental model

| Python `operator` | C# equivalent |
|---|---|
| `operator.add` | `(a, b) => a + b` (no built-in delegate for operators) |
| `itemgetter("name")` | `d => d["name"]` |
| `itemgetter(1)` | `t => t.Item2` |
| `attrgetter("price")` | `p => p.Price` |
| `methodcaller("upper")` | `s => s.ToUpper()` |
| `operator.index(x)` | implicit conversion to an integer type, but only for "true" integers |

The closest .NET idea is a library of pre-built `Func<...>` delegates. There's no BCL equivalent — in C# you always write the lambda. In Python you *can* write the lambda too; `operator` is a slightly faster, more declarative shorthand.

## Operators as functions

Every arithmetic, comparison and logical operator has a function counterpart:

| Expression | Function |
|---|---|
| `a + b`, `a - b`, `a * b` | `add(a, b)`, `sub(a, b)`, `mul(a, b)` |
| `a / b`, `a // b`, `a % b`, `a ** b` | `truediv`, `floordiv`, `mod`, `pow` |
| `-a`, `not a` | `neg(a)`, `not_(a)` |
| `a == b`, `a < b` | `eq(a, b)`, `lt(a, b)` |
| `a & b`, `a \| b` | `and_(a, b)`, `or_(a, b)` |
| `b in a` | `contains(a, b)` (note the order) |
| `a[k]` | `getitem(a, k)` |

Functions whose names would clash with keywords get a trailing underscore: `and_`, `or_`, `not_`.

In [ ]:
import operator as op

print(op.add(2, 3), op.mul(4, 5), op.floordiv(7, 2), op.neg(9))
print(op.lt(1, 2), op.eq("a", "b"), op.contains([1, 2, 3], 2))
print(op.getitem({"x": 10}, "x"), op.or_({1, 2}, {2, 3}))

They shine when you need to **pass an operator to a higher-order function**:

```csharp
var total   = numbers.Aggregate((a, b) => a + b);
var product = numbers.Aggregate((a, b) => a * b);
var sums    = xs.Zip(ys, (a, b) => a + b);
```

In [ ]:
from functools import reduce
from itertools import accumulate

numbers = [1, 2, 3, 4, 5]
print(reduce(op.mul, numbers))              # 120 — Aggregate((a, b) => a * b)
print(list(accumulate(numbers, op.mul)))    # running product
print(list(map(op.add, [1, 2, 3], [10, 20, 30])))   # element-wise: Zip((a, b) => a + b)
print(list(map(op.neg, numbers)))

## `itemgetter` — index or key lookup as a function

`itemgetter(k)` returns a callable that does `obj[k]`. It's the classic `key=` for sorting tuples and dicts.

```csharp
var byPrice = rows.OrderBy(r => r.Item2);
var byName  = dicts.OrderBy(d => d["name"]);
```

In [ ]:
from operator import itemgetter

rows = [("pen", 1.50, 120), ("desk", 89.00, 4), ("lamp", 24.99, 15)]
print(sorted(rows, key=itemgetter(1)))       # by price (index 1)

users = [{"name": "carol", "age": 35}, {"name": "alice", "age": 29}, {"name": "bob", "age": 35}]
print([u["name"] for u in sorted(users, key=itemgetter("name"))])

Pass **several** keys and it returns a tuple — a ready-made composite sort key (`OrderBy(...).ThenBy(...)`):

In [ ]:
by_age_then_name = itemgetter("age", "name")
print(by_age_then_name(users[0]))                 # (35, 'carol') — a tuple
for u in sorted(users, key=by_age_then_name):
    print(u)

It also works anywhere you need "pick these fields": with `map`, `max`/`min`, or slices:

In [ ]:
print(list(map(itemgetter(0), rows)))            # just the names
print(max(rows, key=itemgetter(2)))              # highest stock
print(itemgetter(slice(0, 2))("abcdef"))         # 'ab' — like s[0:2]

## `attrgetter` — attribute access as a function

`attrgetter("price")` does `obj.price`; dotted names reach into nested objects, and multiple names return a tuple.

In [ ]:
from dataclasses import dataclass
from operator import attrgetter

@dataclass
class Supplier:
    city: str

@dataclass
class Product:
    name: str
    price: float
    supplier: Supplier

catalog = [
    Product("lamp", 24.99, Supplier("Oslo")),
    Product("pen", 1.50, Supplier("Bern")),
    Product("desk", 89.00, Supplier("Bern")),
]

print([p.name for p in sorted(catalog, key=attrgetter("price"))])
print([p.name for p in sorted(catalog, key=attrgetter("supplier.city", "name"))])

## `methodcaller` — call a method by name

`methodcaller("name", *args)` returns a callable that does `obj.name(*args)`. Useful with `map` when every item needs the same method call.

In [ ]:
from operator import methodcaller

skus = ["ab-12", "cd-34", "ef-56"]
print(list(map(methodcaller("upper"), skus)))
print(list(map(methodcaller("replace", "-", "_"), skus)))
print(sorted(["Banana", "apple", "Cherry"], key=methodcaller("casefold")))

**Tip:** for a method with no arguments you can often pass the **unbound method** directly: `map(str.upper, skus)` or `key=str.casefold`. `methodcaller` is mainly useful when you need to pass arguments, or when items are of different types that share a method name.

In [ ]:
print(list(map(str.upper, skus)))

## `operator.index` — "give me a real integer"

`operator.index(x)` calls `x.__index__()`: it succeeds for objects that are **genuinely integers** (`int`, `bool`, integer-like types from libraries) and raises `TypeError` for anything else, including `float`. It's what Python uses internally for list indexes and slices.

Contrast with `int(x)`, which happily **truncates** floats and parses strings — often not what you want when validating an index or count.

In [ ]:
from operator import index

print(index(7), index(True))      # 7 1
for value in [3.0, "3"]:
    try:
        index(value)
    except TypeError as e:
        print(f"{type(e).__name__}: {e}")

print(int(3.9), int("3"))         # int() converts — 3.9 becomes 3 silently

Use it when writing your own APIs that accept "an integer or anything integer-like" — similar to how a C# method taking `int` rejects a `double` at compile time:

In [ ]:
def take_first(items, n):
    n = index(n)                  # reject 2.5, accept 2 or True
    return items[:n]

print(take_first("abcdef", 2))
try:
    take_first("abcdef", 2.5)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## A few more handy functions

- `operator.call(f, *args)` (3.11+) — calls `f(*args)`; handy for "run each of these callbacks" with `map`.
- `operator.is_none(x)` / `is_not_none(x)` (3.14+) — the function form of `x is None`, useful as a filter predicate.
- `operator.truth(x)` — same as `bool(x)`.

In [ ]:
callbacks = [lambda: "saved", lambda: "logged"]
print(list(map(op.call, callbacks)))

readings = [12, None, 0, None, 3]
print(list(filter(op.is_not_none, readings)))   # [12, 0, 3] — drops only the Nones
print(list(filter(None, readings)))             # [12, 3]   — drops every falsy value, including 0

## `operator` vs `lambda` — which to use?

| Prefer `operator` when... | Prefer a `lambda` when... |
|---|---|
| the key is a plain index, key or attribute | the key needs a computation (`-x.price`, `len(x.name)`) |
| you need a multi-field key (`itemgetter("a", "b")`) | fields need different directions (asc/desc) |
| passing a raw operator (`op.add`, `op.mul`) | it's a one-off and the lambda reads more clearly to your team |
| you want a picklable callable (`multiprocessing`) | you'd need to nest several `operator` calls |

Both work; `operator` callables are implemented in C and a little faster, but the difference rarely matters. Readability wins.

In [ ]:
# operator shines: simple field access
print([p.name for p in sorted(catalog, key=attrgetter("price"))])

# lambda wins: mixed directions — price descending, then name ascending
print([p.name for p in sorted(catalog, key=lambda p: (-p.price, p.name))])

⚠️ **Gotcha:** `itemgetter` with one argument returns a **single value**, but with two or more it returns a **tuple**. Code that unpacks the result can break if someone later changes the number of fields.

In [ ]:
print(itemgetter(0)(rows[0]))       # 'pen'
print(itemgetter(0, 1)(rows[0]))    # ('pen', 1.5)

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Pass `+` as a function | `(a, b) => a + b` | `operator.add` |
| Product of a sequence | `Aggregate((a, b) => a * b)` | `reduce(operator.mul, xs)` (or `math.prod`) |
| Sort tuples by field 2 | `OrderBy(t => t.Item2)` | `sorted(xs, key=itemgetter(1))` |
| Sort dicts by key | `OrderBy(d => d["k"])` | `sorted(xs, key=itemgetter("k"))` |
| Composite key | `OrderBy(a).ThenBy(b)` | `itemgetter("a", "b")` / `attrgetter("a", "b")` |
| Sort by property | `OrderBy(p => p.Price)` | `sorted(xs, key=attrgetter("price"))` |
| Call a method on each | `Select(s => s.Replace("-", "_"))` | `map(methodcaller("replace", "-", "_"), xs)` |
| Strict integer check | parameter typed as `int` | `operator.index(x)` |
| Not-None filter | `Where(x => x is not null)` | `filter(operator.is_not_none, xs)` (3.14+) |

## Exercises

1. Given `scores = [("alice", 88), ("bob", 95), ("carol", 88)]`, sort by score descending, then name ascending. Which tool did you pick and why?
2. Use `reduce` and `operator.or_` to merge `[{1, 2}, {2, 3}, {5}]` into one set.
3. Given a list of `Product` objects (as in this notebook), use `attrgetter` to find the cheapest product and to print each product's supplier city.
4. Use `map` with `operator.mul` to compute the dot product of `[1, 2, 3]` and `[4, 5, 6]`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import operator as op
from functools import reduce
from operator import attrgetter
from dataclasses import dataclass

# 1 — mixed directions: a lambda is clearest
scores = [("alice", 88), ("bob", 95), ("carol", 88)]
print(sorted(scores, key=lambda s: (-s[1], s[0])))
# [('bob', 95), ('alice', 88), ('carol', 88)]

# 2 — | on sets is union
print(reduce(op.or_, [{1, 2}, {2, 3}, {5}]))   # {1, 2, 3, 5}

# 3
@dataclass
class Supplier:
    city: str

@dataclass
class Product:
    name: str
    price: float
    supplier: Supplier

catalog = [Product("lamp", 24.99, Supplier("Oslo")), Product("pen", 1.50, Supplier("Bern"))]
print(min(catalog, key=attrgetter("price")).name)          # pen
print(list(map(attrgetter("supplier.city"), catalog)))     # ['Oslo', 'Bern']

# 4 — map with two iterables pairs them up, like Zip
print(sum(map(op.mul, [1, 2, 3], [4, 5, 6])))   # 32
```

</details>